# DEE — Notebook B: Estructura del modo localizado + Espectro primordial

## Tres tests interpretativos sobre el sustrato

**Test 4**: Caracterización geométrica del autovector localizado del defecto. ¿Es un dímero (dos sitios), dipolo elástico, o estructura más compleja?

**Test 5**: Robustez de la localización al cambiar N y seed. ¿La participación baja persiste?

**Test 6**: Espectro de potencias P(k) del cristal post-aging. ¿Qué n_s da el sustrato? Contraste con la predicción inflacionaria n_s ≈ 0.96.

**Tiempo estimado en Colab Pro**: 30-60 minutos.

**Outputs**:
- Plots 3D del autovector localizado (Test 4)
- Tabla robustez participación con N y seed (Test 5)
- Espectro P(k) y n_s del cristal (Test 6)

In [ ]:
# ============================================================
# Imports
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
from scipy.spatial import cKDTree
import time
import os

os.makedirs('plots_estructura_v5', exist_ok=True)
np.random.seed(42)
print('Setup OK')

In [ ]:
# ============================================================
# Funciones (idénticas a Notebook A para que sean autocontenidas)
# ============================================================

def grid_perturbed_T3(N_target, jitter_fraction=0.10, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def build_laplacian_T3(points, k_neighbors=30, alpha=2.0, defect_center=None,
                       defect_radius=0.15, defect_factor=1.0, L=1.0):
    N = len(points)
    images = []
    image_to_orig = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                images.append(points + np.array([dx*L, dy*L, dz*L]))
                image_to_orig.extend([(i, dx, dy, dz) for i in range(N)])
    images_all = np.concatenate(images, axis=0)
    image_to_orig = np.array([io[0] for io in image_to_orig])
    tree = cKDTree(images_all)
    rows, cols, vals = [], [], []
    if defect_center is not None:
        diff = points - defect_center
        diff = diff - L * np.round(diff / L)
        dist_to_center = np.linalg.norm(diff, axis=1)
        in_defect = dist_to_center < defect_radius
    else:
        in_defect = np.zeros(N, dtype=bool)
    for i in range(N):
        dists, idxs = tree.query(points[i], k=k_neighbors+1)
        valid = ~((image_to_orig[idxs] == i) & (dists < 1e-12))
        v_idx = np.where(valid)[0][:k_neighbors]
        for vi in v_idx:
            j = image_to_orig[idxs[vi]]
            d = dists[vi]
            if d > 1e-10:
                weight = 1.0 / d**alpha
                if in_defect[i] or in_defect[j]:
                    weight *= defect_factor
                rows.append(i); cols.append(j); vals.append(weight)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    L_op = diags(degrees) - W
    return L_op, in_defect

def find_principal_defect_mode(L_op, in_defect, n_modes=300, tol=1e-8):
    eigs, vecs = eigsh(L_op, k=n_modes, which='SM', tol=tol)
    order = np.argsort(eigs); eigs = eigs[order]; vecs = vecs[:, order]
    nonzero = eigs > 1e-3
    eigs_nz = eigs[nonzero]; vecs_nz = vecs[:, nonzero]
    omegas = np.sqrt(eigs_nz)
    n_defect = in_defect.sum(); n_total = len(in_defect)
    fill_frac = n_defect / n_total
    concentration = []
    for k in range(len(eigs_nz)):
        psi2 = vecs_nz[:, k]**2
        norm = psi2.sum()
        if norm > 1e-12:
            concentration.append(psi2[in_defect].sum() / norm / fill_frac)
        else:
            concentration.append(0.0)
    concentration = np.array(concentration)
    candidates = np.where(concentration > 5.0)[0]
    idx_principal = candidates[0] if len(candidates) > 0 else np.argmax(concentration)
    return {
        'omegas_all': omegas, 'eigs_all': eigs_nz, 'vecs_all': vecs_nz,
        'concentration': concentration, 'idx_principal': idx_principal,
        'omega_def': omegas[idx_principal], 'vec_def': vecs_nz[:, idx_principal],
    }

print('Funciones cargadas')

## Test 4 — Caracterización geométrica del autovector localizado

Construir un cristal con inclusión blanda 3D ×0.1 (defect_factor=0.1, radius=0.15) y diagonalizar. Tomar el modo principal del defecto (ω_def ≈ 9.66 esperado) y analizar:

- **Distribución espacial 3D**: dónde se concentra ψ.
- **Top sitios por |ψ|²**: ¿son 2, 3, 4, más?
- **Signos de ψ en los sitios principales**: ¿mismo signo (modo s-like) u opuestos (modo p-like / dipolo)?
- **Distancia entre sitios principales**: ¿hay una escala bien definida?
- **Perfil radial desde el centroide**: ¿exponencial puro o con oscilaciones?

In [ ]:
# ============================================================
# Test 4: estructura del modo localizado a N=1331, seed=42
# ============================================================

N_target = 1331
seed = 42
defect_center = np.array([0.5, 0.5, 0.5])
defect_radius = 0.15
defect_factor = 0.1

print(f'Construyendo cristal N={N_target} con defecto blando ×{defect_factor}...')
t0 = time.time()
points, N = grid_perturbed_T3(N_target, 0.10, seed=seed)
L_op, in_defect = build_laplacian_T3(
    points, k_neighbors=30, alpha=2.0,
    defect_center=defect_center, defect_radius=defect_radius,
    defect_factor=defect_factor, L=1.0
)
print(f'Construido en {time.time()-t0:.1f}s. Nodos en defecto: {in_defect.sum()}/{N}')

print('Diagonalizando...')
t0 = time.time()
mode = find_principal_defect_mode(L_op, in_defect, n_modes=300)
print(f'Diagonalizado en {time.time()-t0:.1f}s')

print(f'\nω_def = {mode["omega_def"]:.4f}')
print(f'Concentración relativa al baseline: {mode["concentration"][mode["idx_principal"]]:.1f}x')

# Participation ratio
psi = mode['vec_def']
psi2 = psi**2
IPR = (psi2**2).sum() / (psi2.sum())**2
participation = 1.0 / IPR
print(f'Participation ratio: {participation:.2f}')

In [ ]:
# ============================================================
# Test 4 — Análisis de top sitios y signos
# ============================================================

psi = mode['vec_def']
psi2 = psi**2

# Ordenar sitios por |ψ|²
order = np.argsort(psi2)[::-1]
top10 = order[:10]

print('=== TOP 10 SITIOS POR |ψ|² ===\n')
print(f'{"rank":>4s} | {"idx":>5s} | {"posición":^28s} | {"|ψ|²":>10s} | {"signo ψ":>8s} | {"d_centro":>9s} | {"in_def":>6s}')
print('-'*100)
for rank, idx in enumerate(top10):
    pos = points[idx]
    diff = pos - defect_center
    diff = diff - 1.0 * np.round(diff)
    d = np.linalg.norm(diff)
    sign = '+' if psi[idx] > 0 else '-'
    print(f'{rank+1:>4d} | {idx:>5d} | ({pos[0]:.3f},{pos[1]:.3f},{pos[2]:.3f}) | '
          f'{psi2[idx]:>10.6f} | {sign:>8s} | {d:>9.4f} | {str(in_defect[idx]):>6s}')

# Suma de los top 2, top 4, top 10
total = psi2.sum()
print(f'\n=== Concentración acumulada ===')
print(f'Top 1:  {psi2[top10[0]]/total*100:.1f}% del total')
print(f'Top 2:  {psi2[top10[:2]].sum()/total*100:.1f}% del total')
print(f'Top 4:  {psi2[top10[:4]].sum()/total*100:.1f}% del total')
print(f'Top 10: {psi2[top10[:10]].sum()/total*100:.1f}% del total')

# Distancia entre top 1 y top 2
p1, p2 = points[top10[0]], points[top10[1]]
diff_12 = p1 - p2
diff_12 = diff_12 - 1.0 * np.round(diff_12)
d_12 = np.linalg.norm(diff_12)
print(f'\nDistancia entre top1 y top2: {d_12:.4f}')
print(f'(comparar con radio defecto = {defect_radius:.3f})')

# Signos
sign_top4 = np.sign(psi[top10[:4]])
print(f'\nSignos de los top 4: {sign_top4}')
if (sign_top4[0] != sign_top4[1]):
    print('→ Top 1 y Top 2 tienen SIGNOS OPUESTOS: estructura tipo dipolo elástico')
elif np.all(sign_top4 == sign_top4[0]):
    print('→ Todos los top sitios tienen MISMO signo: estructura s-like (estado ligado simétrico)')
else:
    print('→ Estructura mixta de signos: posible multipolo')

In [ ]:
# ============================================================
# Test 4 — Visualización 3D del autovector
# ============================================================

fig = plt.figure(figsize=(15, 6))

# Plot 3D: tamaño y color = |ψ|²
ax = fig.add_subplot(121, projection='3d')
# Filtrar sitios significativos (top 5% por |ψ|²)
threshold = np.percentile(psi2, 95)
mask_sig = psi2 > threshold
scatter_size = psi2[mask_sig] / psi2.max() * 200 + 10
scatter_color = psi[mask_sig]  # signo
sc = ax.scatter(points[mask_sig, 0], points[mask_sig, 1], points[mask_sig, 2],
                s=scatter_size, c=scatter_color, cmap='RdBu_r', alpha=0.8,
                edgecolors='black', linewidths=0.3)
# Centro del defecto
ax.scatter([defect_center[0]], [defect_center[1]], [defect_center[2]],
           s=300, c='black', marker='x', linewidths=3, label='Centro defecto')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title(f'Autovector ψ del modo principal (top 5% sitios)\n'
             f'ω_def={mode["omega_def"]:.3f}, participation={participation:.1f}')
plt.colorbar(sc, ax=ax, label='ψ (signo)', shrink=0.6)
ax.legend()

# Plot perfil radial desde el centro del defecto
ax2 = fig.add_subplot(122)
diff = points - defect_center
diff = diff - 1.0 * np.round(diff)
r = np.linalg.norm(diff, axis=1)
# Bins radiales
r_max = 0.5
n_bins = 30
r_bins = np.linspace(0, r_max, n_bins+1)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
rho_r = np.zeros(n_bins)
counts = np.zeros(n_bins)
for i in range(N):
    bin_idx = int(r[i] / r_max * n_bins)
    if 0 <= bin_idx < n_bins:
        rho_r[bin_idx] += psi2[i]
        counts[bin_idx] += 1
# Densidad por sitio promedio en cada cáscara
rho_r_per_site = np.where(counts > 0, rho_r / np.maximum(counts, 1), 0)

ax2.semilogy(r_centers, rho_r_per_site + 1e-12, 'o-', markersize=8, color='C3')
ax2.axvline(defect_radius, ls='--', color='red', label=f'Borde defecto r={defect_radius}')
ax2.set_xlabel('r (distancia al centro del defecto)')
ax2.set_ylabel('|ψ|² promedio por sitio (log)')
ax2.set_title('Perfil radial del modo localizado')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plots_estructura_v5/test4_autovector_3D.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot guardado: plots_estructura_v5/test4_autovector_3D.png')

## Test 5 — Robustez de la localización al variar N y seed

In [ ]:
# ============================================================
# Test 5 — Sweep de N y seeds para participation y ω_def
# ============================================================

Ns = [729, 1331, 3375]  # n_side = 9, 11, 15
seeds = [42, 100, 200, 333, 500]

robust_results = []

for N_target in Ns:
    for seed in seeds:
        t0 = time.time()
        points, N = grid_perturbed_T3(N_target, 0.10, seed=seed)
        L_op, in_defect = build_laplacian_T3(
            points, k_neighbors=30, alpha=2.0,
            defect_center=np.array([0.5, 0.5, 0.5]),
            defect_radius=0.15, defect_factor=0.1, L=1.0
        )
        n_modes = min(300, N - 5)
        mode = find_principal_defect_mode(L_op, in_defect, n_modes=n_modes)
        
        psi2 = mode['vec_def']**2
        IPR = (psi2**2).sum() / (psi2.sum())**2
        participation = 1.0 / IPR
        
        robust_results.append({
            'N': N, 'seed': seed,
            'omega_def': mode['omega_def'],
            'participation': participation,
        })
        print(f'N={N}, seed={seed}: ω_def={mode["omega_def"]:.4f}, '
              f'part={participation:.2f}, t={time.time()-t0:.1f}s')

# Resumen
print('\n=== RESUMEN Test 5 ===')
print(f'{"N":>6s} | {"ω_def medio":>15s} | {"part. media":>15s} | {"part. min":>10s} | {"part. max":>10s}')
for N_target in Ns:
    subset = [r for r in robust_results if r['N'] == N_target or 
              (N_target == 729 and r['N'] == 729) or
              (N_target == 1331 and r['N'] == 1331) or
              (N_target == 3375 and r['N'] == 3375)]
    om = np.array([r['omega_def'] for r in subset])
    pa = np.array([r['participation'] for r in subset])
    if len(om) > 0:
        N_actual = subset[0]['N']
        print(f'{N_actual:>6d} | {np.mean(om):>7.4f} ± {np.std(om):>5.4f} | '
              f'{np.mean(pa):>7.2f} ± {np.std(pa):>5.2f} | {pa.min():>10.2f} | {pa.max():>10.2f}')

In [ ]:
# ============================================================
# Plot Test 5
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for N_target in Ns:
    subset = [r for r in robust_results if r['N'] == N_target]
    if not subset:
        for n_test in [N_target]:
            subset = [r for r in robust_results if abs(r['N'] - n_test) < 100]
    if subset:
        N_actual = subset[0]['N']
        seeds_x = [r['seed'] for r in subset]
        ome = [r['omega_def'] for r in subset]
        ax.plot(seeds_x, ome, 'o', markersize=10, label=f'N={N_actual}', alpha=0.7)
ax.set_xlabel('seed')
ax.set_ylabel('ω_def')
ax.set_title('Robustez de ω_def al cambiar seed')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for N_target in Ns:
    subset = [r for r in robust_results if r['N'] == N_target]
    if not subset:
        for n_test in [N_target]:
            subset = [r for r in robust_results if abs(r['N'] - n_test) < 100]
    if subset:
        N_actual = subset[0]['N']
        seeds_x = [r['seed'] for r in subset]
        pa = [r['participation'] for r in subset]
        ax.plot(seeds_x, pa, 's', markersize=10, label=f'N={N_actual}', alpha=0.7)
ax.set_xlabel('seed')
ax.set_ylabel('Participation ratio')
ax.set_title('Robustez de la localización al cambiar seed')
ax.set_yscale('log')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plots_estructura_v5/test5_robustez.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: plots_estructura_v5/test5_robustez.png')

## Test 6 — Espectro de potencias P(k) del cristal post-aging

**Pregunta**: ¿el sustrato DEE produce un espectro de fluctuaciones casi-invariante de escala (n_s ≈ 0.96, predicción inflacionaria) sin invocar inflación?

**Protocolo**:
1. Construir cristal T³ con jitter 0.10 (estado fundamental relajado, equivalente a post-aging para nuestros propósitos).
2. Calcular el campo de fluctuaciones φ(x) en cada nodo. Como aproximación al estado térmico de baja temperatura, usamos las amplitudes de los modos de baja energía con peso ∝ 1/ω (equipartición fonónica).
3. Hacer FFT y calcular P(k) = ⟨|φ̃(k)|²⟩.
4. Promediar P(k) en cáscaras de |k| (binning radial).
5. Ajustar P(k) ∝ k^(n_s−1) en el rango lineal.

**Caveats importantes**:
- Este test NO es un cálculo cosmológico completo. Es una primera mirada al espectro intrínseco del sustrato.
- Si el cristal está en un estado térmico clásico, equipartición da P(k) ∝ 1/ω² ∝ 1/k² para fonones acústicos, lo cual es n_s = -1 (rojo extremo, Harrison-Zel'dovich es n_s = 1).
- La predicción inflacionaria n_s ≈ 0.96 viene de **amplificación cuántica** de fluctuaciones de vacío durante inflación, no de equipartición clásica.
- Lo que medimos acá es el espectro **clásico** del sustrato. El resultado servirá como punto de partida; si difiere mucho de n_s = 0.96, eso confirma que se necesitaría dinámica adicional (anharmónica o fuera de equilibrio) para reproducir inflación.

In [ ]:
# ============================================================
# Test 6 — Cristal sin defecto, espectro térmico
# ============================================================

N_target = 3375  # n_side=15 para tener suficientes modos
seed = 42

print(f'Construyendo cristal limpio N={N_target}...')
t0 = time.time()
points, N = grid_perturbed_T3(N_target, 0.10, seed=seed)
L_op, _ = build_laplacian_T3(
    points, k_neighbors=30, alpha=2.0,
    defect_center=None, L=1.0
)
print(f'Construido en {time.time()-t0:.1f}s')

# Diagonalizar muchos modos
n_modes = min(800, N - 5)
print(f'Diagonalizando {n_modes} modos...')
t0 = time.time()
eigs, vecs = eigsh(L_op, k=n_modes, which='SM', tol=1e-8)
order = np.argsort(eigs)
eigs = eigs[order]; vecs = vecs[:, order]
nonzero = eigs > 1e-3
eigs_nz = eigs[nonzero]; vecs_nz = vecs[:, nonzero]
omegas = np.sqrt(eigs_nz)
print(f'Diagonalizado en {time.time()-t0:.1f}s. Modos no-cero: {len(omegas)}')

In [ ]:
# ============================================================
# Construir el campo φ(x) en estado térmico (equipartición clásica)
# ============================================================

# En el estado térmico clásico a temperatura T_eff, cada modo n tiene
# amplitud cuadrática promedio ⟨|q_n|²⟩ = T / ω_n²
# φ(x) = sum_n q_n · ψ_n(x)
# Generamos una realización aleatoria con varianzas k_B T / ω_n²

T_eff = 1.0  # temperatura en unidades sim
rng = np.random.default_rng(seed=42)

# Coeficientes con varianza T/ω²
amplitudes = rng.normal(0, np.sqrt(T_eff / omegas**2))

# Campo φ en cada sitio
phi_field = vecs_nz @ amplitudes

print(f'Campo φ construido. Estadísticas:')
print(f'  ⟨φ⟩ = {phi_field.mean():.4f}')
print(f'  std(φ) = {phi_field.std():.4f}')
print(f'  min/max = {phi_field.min():.4f} / {phi_field.max():.4f}')

In [ ]:
# ============================================================
# Calcular espectro P(k) por método espectral directo
# (usando los modos del cristal en lugar de FFT en grid)
# ============================================================

# Como nuestros nodos están en una grilla con jitter, no podemos hacer FFT directa.
# Usamos el método más correcto: los autovectores del Laplaciano SON los modos de
# Fourier en T³ (verificado en SIM 10 con peso >99.9% en |n|²=cte).
# El autovalor λ_n = (2π|n|/L)² × C_kernel, donde C_kernel es una constante del kernel.
# Por lo tanto |k_n|² = λ_n / C_kernel, y |k_n| ∝ √λ_n = ω_n.
# Es decir, ω_n ES proporcional a |k_n| para fonones acústicos.

# Definimos k_n = ω_n (constante de proporcionalidad absorbida en normalización)
# La densidad de potencia P(k_n) = |amplitude_n|² evaluado en k_n

k_values = omegas.copy()  # |k| ∝ ω
P_values = amplitudes**2

# Binning logarítmico en k para obtener P(k) suave
k_min = k_values.min() * 1.05
k_max = k_values.max() * 0.95
n_bins = 25
k_bins = np.logspace(np.log10(k_min), np.log10(k_max), n_bins+1)
k_centers = np.sqrt(k_bins[:-1] * k_bins[1:])  # centro geométrico
P_binned = np.zeros(n_bins)
counts = np.zeros(n_bins)

for i, k in enumerate(k_values):
    bin_idx = np.searchsorted(k_bins, k) - 1
    if 0 <= bin_idx < n_bins:
        P_binned[bin_idx] += P_values[i]
        counts[bin_idx] += 1

P_avg = np.where(counts > 0, P_binned / np.maximum(counts, 1), np.nan)
valid_bins = (counts >= 3) & ~np.isnan(P_avg) & (P_avg > 0)

# Ajuste lineal en log-log: log(P) = (n_s - 1) log(k) + const
if valid_bins.sum() >= 5:
    log_k = np.log10(k_centers[valid_bins])
    log_P = np.log10(P_avg[valid_bins])
    slope, intercept = np.polyfit(log_k, log_P, 1)
    n_s = slope + 1.0
    print(f'\n=== AJUSTE P(k) ∝ k^(n_s − 1) ===')
    print(f'  Pendiente log-log:        {slope:+.4f}')
    print(f'  n_s implicado:            {n_s:+.4f}')
    print(f'  Predicción inflacionaria: ≈ 0.96')
    print(f'  Predicción Harrison-Zeldovich: 1.00')
    print(f'  Equipartición fonones acústicos: −1.00')
else:
    n_s = np.nan
    print('No hay suficientes bins válidos para ajuste')

In [ ]:
# ============================================================
# Plot espectro P(k)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: P(k) crudo y binned
ax = axes[0]
ax.scatter(k_values, P_values, s=8, alpha=0.3, color='gray', label='Modos individuales')
ax.plot(k_centers[valid_bins], P_avg[valid_bins], 'o-', color='C3',
        markersize=10, linewidth=2, label='Binned (centro geom.)')
if not np.isnan(n_s):
    fit_x = np.array([k_centers[valid_bins].min(), k_centers[valid_bins].max()])
    fit_y = 10**intercept * fit_x**slope
    ax.plot(fit_x, fit_y, 'k--', linewidth=2, label=f'Ajuste: n_s = {n_s:+.3f}')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('k (∝ ω)')
ax.set_ylabel('P(k)')
ax.set_title('Espectro de potencias del cristal en estado térmico')
ax.legend(); ax.grid(alpha=0.3, which='both')

# Plot 2: comparación con benchmarks
ax = axes[1]
if not np.isnan(n_s):
    benchmarks = {
        'DEE medido': n_s,
        'Inflación slow-roll (Planck)': 0.965,
        'Harrison-Zeldovich': 1.0,
        'Equipartición acústica (T_eff const.)': -1.0,
        'White noise': 0.0,
    }
    colors = ['C3', 'green', 'blue', 'orange', 'gray']
    y_pos = np.arange(len(benchmarks))
    ns_vals = list(benchmarks.values())
    ax.barh(y_pos, ns_vals, color=colors, alpha=0.7, edgecolor='black')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(list(benchmarks.keys()))
    ax.axvline(0.96, ls='--', color='green', alpha=0.5, label='Banda observacional')
    ax.set_xlabel('n_s')
    ax.set_title('Comparación con benchmarks')
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('plots_estructura_v5/test6_espectro_primordial.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: plots_estructura_v5/test6_espectro_primordial.png')

In [ ]:
# ============================================================
# Síntesis final
# ============================================================

print('='*70)
print('SÍNTESIS — Notebook B: estructura del modo localizado + espectro')
print('='*70)

print(f'\n--- Test 4 (estructura geométrica del modo localizado) ---')
print(f'  ω_def = {mode["omega_def"]:.4f}')
print(f'  Participation ratio = {participation:.2f}')
print(f'  Top 2 sitios concentran {psi2[top10[:2]].sum()/total*100:.1f}% del total')
print(f'  Distancia top1-top2 = {d_12:.4f}')
print(f'  Signos top4 = {sign_top4}')
if sign_top4[0] != sign_top4[1]:
    print(f'  → INTERPRETACIÓN: dipolo elástico (signos opuestos)')
elif np.all(sign_top4 == sign_top4[0]):
    print(f'  → INTERPRETACIÓN: estado ligado s-like (mismo signo)')
else:
    print(f'  → INTERPRETACIÓN: estructura mixta')

print(f'\n--- Test 5 (robustez al cambiar N y seed) ---')
print(f'  Total corridas: {len(robust_results)}')
all_part = [r["participation"] for r in robust_results]
print(f'  Participation: rango [{min(all_part):.2f}, {max(all_part):.2f}]')
print(f'  Mediana participation = {np.median(all_part):.2f}')
if np.median(all_part) < 10:
    print(f'  → Localización extrema CONFIRMADA en sweep')
else:
    print(f'  → Localización menos extrema en algunas corridas')

print(f'\n--- Test 6 (espectro primordial) ---')
if not np.isnan(n_s):
    print(f'  n_s medido = {n_s:+.4f}')
    if 0.85 < n_s < 1.05:
        print(f'  → CASI-INVARIANTE de escala (sorprendente)')
    elif n_s < -0.5:
        print(f'  → ESPECTRO ROJO PROFUNDO (consistente con equipartición acústica)')
    else:
        print(f'  → Espectro intermedio (entre acústico y plano)')
    print(f'  Caveat: este es el espectro térmico CLÁSICO del sustrato; la')
    print(f'          predicción inflacionaria viene de amplificación cuántica.')

# Guardar resumen
import pickle
with open('plots_estructura_v5/results_raw.pkl', 'wb') as f:
    pickle.dump({
        'test4': {
            'omega_def': mode['omega_def'],
            'participation': participation,
            'top10_idx': top10.tolist(),
            'top10_psi': psi[top10].tolist(),
            'top10_psi2': psi2[top10].tolist(),
            'd_12': d_12,
            'sign_top4': sign_top4.tolist(),
        },
        'test5': robust_results,
        'test6': {
            'k_values': k_values.tolist() if hasattr(k_values, 'tolist') else list(k_values),
            'P_values': P_values.tolist() if hasattr(P_values, 'tolist') else list(P_values),
            'k_centers_binned': k_centers.tolist(),
            'P_avg_binned': P_avg.tolist(),
            'n_s': n_s,
        }
    }, f)
print('\nResultados raw guardados: plots_estructura_v5/results_raw.pkl')